In [1]:
import pickle
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
import pandas as pd 
df = pd.read_csv("../Resultados_Finales_TFM.csv")
df["ratio"] = df["ratio"].fillna(0)
# df = df[df["n_classes"] != 2]

In [3]:
df

,Model,Accuracy,Precision,Recall,F1,dataset,ratio,sampler,seed,n_classes,nrows,nfeat
0,Random Forest,0.989565,0.494783,0.500000,0.497378,abalone-20_vs_8-9-10,0.0,Original,0,2,1916,10
1,XGBoost,0.991304,0.995645,0.583333,0.640670,abalone-20_vs_8-9-10,0.0,Original,0,2,1916,10
2,LightGBM,0.993043,0.872373,0.749121,0.798246,abalone-20_vs_8-9-10,0.0,Original,0,2,1916,10
3,Random Forest,0.984348,0.691418,0.909637,0.759179,abalone-20_vs_8-9-10,0.6,smote,0,2,1916,10
4,XGBoost,0.977391,0.646163,0.906122,0.711624,abalone-20_vs_8-9-10,0.6,smote,0,2,1916,10
...,...,...,...,...,...,...,...,...,...,...,...,...
22138,XGBoost,0.979821,0.851356,0.828693,0.839613,yeast6,0.8,borderline,29,2,1484,8
22139,LightGBM,0.984305,0.910906,0.831013,0.866324,yeast6,0.8,borderline,29,2,1484,8
22140,Random Forest,0.979821,0.868088,0.796520,0.828131,yeast6,1.0,borderline,29,2,1484,8
22141,XGBoost,0.979821,0.851356,0.828693,0.839613,yeast6,1.0,borderline,29,2,1484,8


In [4]:
(df["F1"] == 1).sum()

np.int64(753)

In [5]:
import pandas as pd
import plotly.express as px

# 1. PASO CRUCIAL: Seleccionar el mejor ratio para cada sampler en cada dataset/modelo
# Primero promediamos las 30 semillas para cada combinación
df_seeds_mean = df.groupby(['dataset', 'Model', 'sampler', 'ratio'])['F1'].mean().reset_index()

# Ahora buscamos cuál es el ratio que dio el F1 más alto para cada Sampler/Dataset/Modelo
idx_best_ratio = df_seeds_mean.groupby(['dataset', 'Model', 'sampler'])['F1'].idxmax()
df_best_configs = df_seeds_mean.loc[idx_best_ratio].reset_index(drop=True)

# 2. PROMEDIO ENTRE MODELOS (RF, XGB, LGBM)
# Para tener un único valor por Sampler y Dataset
df_agg = df_best_configs.groupby(['dataset', 'sampler'])['F1'].mean().reset_index()

# 3. CÁLCULO DEL RANKING POR DATASET
# (El rango 1 es para el F1 más alto)
df_agg['rank'] = df_agg.groupby('dataset')['F1'].rank(method='min', ascending=False)

# 4. AGREGACIÓN PARA EL GRÁFICO (Rango medio global)
df_plot = df_agg.groupby('sampler')['rank'].agg(['mean', 'sem']).reset_index()
df_plot = df_plot.sort_values('mean')

# --- VISUALIZACIÓN ---
fig = px.bar(
    df_plot,
    x='sampler',
    y='mean',
    error_y='sem',
    color='sampler',
    title="Ranking Medio de Samplers (Mejor Ratio seleccionado)",
    labels={'mean': 'Rango Medio (Menor es mejor)', 'sampler': 'Técnica de Muestreo'},
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.update_layout(
    yaxis=dict(autorange="reversed", title="Rango Medio (1º es el top)"),
    width=1000, height=600,
    showlegend=False
)

fig.show()

# --- PRINT DE RESULTADOS DETALLADOS ---
print("\n" + "="*50)
print("RANKING BASADO EN EL MEJOR RATIO PROMEDIO (30 SEMILLAS)")
print("="*50)

df_ranking_detail = df_agg.sort_values(['dataset', 'rank'])

for dataset in df_ranking_detail['dataset'].unique():
    print(f"\n📊 Dataset: {dataset}")
    subset = df_ranking_detail[df_ranking_detail['dataset'] == dataset]
    
    # Recuperamos qué ratio fue el ganador para informar en el print
    for _, row in subset.iterrows():
        # Buscamos el ratio que usamos (promediando entre los 3 modelos para ese sampler)
        ratio_info = df_best_configs[
            (df_best_configs['dataset'] == dataset) & 
            (df_best_configs['sampler'] == row['sampler'])
        ]['ratio'].unique()
        
        ratios_str = "/".join(map(str, ratio_info))
        print(f"  {int(row['rank'])}º - {row['sampler']:<15} | F1 Medio: {row['F1']:.4f} (Ratios usados: {ratios_str})")


RANKING BASADO EN EL MEJOR RATIO PROMEDIO (30 SEMILLAS)

📊 Dataset: abalone-20_vs_8-9-10
  1º - borderline      | F1 Medio: 0.6536 (Ratios usados: 1.0/0.6)
  2º - corsmote        | F1 Medio: 0.6511 (Ratios usados: 1.0/0.6/0.8)
  3º - smote           | F1 Medio: 0.6503 (Ratios usados: 0.8/0.6)
  4º - adasyn          | F1 Medio: 0.6437 (Ratios usados: 0.8/1.0)
  5º - Original        | F1 Medio: 0.5846 (Ratios usados: 0.0)

📊 Dataset: bank_account_fraud
  1º - borderline      | F1 Medio: 0.5483 (Ratios usados: 1.0/0.8)
  2º - adasyn          | F1 Medio: 0.5479 (Ratios usados: 0.6/0.8)
  3º - smote           | F1 Medio: 0.5463 (Ratios usados: 0.6/0.8)
  4º - corsmote        | F1 Medio: 0.5421 (Ratios usados: 1.0/0.8)
  5º - Original        | F1 Medio: 0.5186 (Ratios usados: 0.0)

📊 Dataset: cicids2017
  1º - corsmote        | F1 Medio: 0.9932 (Ratios usados: 0.6/1.0)
  2º - smote           | F1 Medio: 0.9930 (Ratios usados: 0.6/0.8)
  3º - Original        | F1 Medio: 0.9929 (Ratios usados

In [6]:
df_agg = df.groupby(['dataset', 'sampler'])['F1'].mean().reset_index()
df_agg['rank'] = df_agg.groupby('dataset')['F1'].rank(method='min', ascending=False)

df_plot = df_agg.groupby('sampler')['rank'].agg(['mean', 'sem']).reset_index()

df_plot = df_plot.sort_values('mean')

fig = px.bar(
    df_plot,
    x='sampler',
    y='mean',
    error_y='sem',               
    color='sampler',                
    color_discrete_sequence=px.colors.qualitative.Plotly,
    labels={'mean': 'Rango medio', 'sampler': 'Sampler'}
)

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    width=1000, height=600,           
    coloraxis_showscale=False         
)

fig.update_traces(error_y=dict(width=10, thickness=1.5))

fig.show()
fig.write_image("../Comparativa_RangoMedio_Sampler.pdf", format="pdf")

print("\n" + "="*40)
print("RANKING DETALLADO POR DATASET")
print("="*40)

# Ordenamos para que la lectura sea lógica: primero por dataset y luego por posición en el ranking
df_ranking_detail = df_agg.sort_values(['dataset', 'rank'])

for dataset in df_ranking_detail['dataset'].unique():
    print(f"\n📊 Dataset: {dataset}")
    subset = df_ranking_detail[df_ranking_detail['dataset'] == dataset]
    
    for _, row in subset.iterrows():
        # Usamos .format o f-strings para alinear un poco el texto
        print(f"  {int(row['rank'])}º - {row['sampler']} (F1: {row['F1']:.4f})")


RANKING DETALLADO POR DATASET

📊 Dataset: abalone-20_vs_8-9-10
  1º - borderline (F1: 0.6492)
  2º - corsmote (F1: 0.6492)
  3º - smote (F1: 0.6460)
  4º - adasyn (F1: 0.6398)
  5º - Original (F1: 0.5846)

📊 Dataset: bank_account_fraud
  1º - borderline (F1: 0.5470)
  2º - adasyn (F1: 0.5456)
  3º - smote (F1: 0.5452)
  4º - corsmote (F1: 0.5400)
  5º - Original (F1: 0.5186)

📊 Dataset: cicids2017
  1º - corsmote (F1: 0.9931)
  2º - Original (F1: 0.9929)
  3º - smote (F1: 0.9929)
  4º - borderline (F1: 0.9926)
  5º - adasyn (F1: 0.9905)

📊 Dataset: cleveland_0_vs_4
  1º - adasyn (F1: 0.8004)
  2º - smote (F1: 0.7975)
  3º - borderline (F1: 0.7857)
  4º - corsmote (F1: 0.7633)
  5º - Original (F1: 0.7068)

📊 Dataset: credit
  1º - corsmote (F1: 0.8951)
  2º - borderline (F1: 0.8909)
  3º - smote (F1: 0.8864)
  4º - adasyn (F1: 0.8756)
  5º - Original (F1: 0.8719)

📊 Dataset: diabetes
  1º - Original (F1: 0.7251)
  2º - corsmote (F1: 0.7214)
  3º - smote (F1: 0.7203)
  4º - borderline (

In [7]:
fig = px.box(df, x="ratio", y="F1", color="sampler")
fig.show()

In [14]:
# --- SEPARACIÓN EN DOS GRÁFICAS ---

# 1. Lista de datasets para la gráfica con ZOOM (Valores muy altos/apretados)
datasets_ultra = ['cicids2017', 'occupancy']
datasets_zoom = [ 'litnet', 'unsw', 'vowel0']

# Filtramos los dos dataframes
df_ultra = df[df['dataset'].isin(datasets_ultra)]
df_zoom = df[df['dataset'].isin(datasets_zoom)]
df_resto = df[~df['dataset'].isin(datasets_zoom + datasets_ultra)]

# --- GRÁFICA 1: ZOOM (Alta Precisión) ---
fig_zoom = px.box(
    df_zoom, 
    y="F1", 
    color="sampler", 
    x="dataset",
    title="Comparativa F1: Datasets de Alta Precisión (Zoom 0.90 - 1.0)"
)
# Ajustamos el rango del eje Y para que se vean las diferencias
fig_zoom.update_yaxes(range=[0.90, 1.005]) 
fig_zoom.show()
fig_zoom.write_image("../ComparativaF1_Zoom.pdf", format="pdf", width=1200, height=600)



In [15]:
# --- GRÁFICA A: ULTRA-PRECISIÓN (0.98 a 1.0) ---
fig_ultra = px.box(df_ultra, x="dataset", y="F1", color="sampler", 
                   title="Detalle Máximo: F1 entre 0.98 y 1.0")
fig_ultra.update_yaxes(range=[0.98, 1.001]) # El 1.001 es para que no se corte la línea superior
fig_ultra.show()
fig_ultra.write_image("../F1_1_Ultra_Precision.pdf", width=1000, height=600)

In [16]:
# --- GRÁFICA 2: RESTO (Rango Estándar) ---
fig_resto = px.box(
    df_resto, 
    y="F1", 
    color="sampler", 
    x="dataset",
    title="Comparativa F1: Datasets Rango Estándar"
)
# Mantenemos el rango completo (0 a 1)
fig_resto.update_yaxes(range=[0.45, 1.005]) 
fig_resto.show()
fig_resto.write_image("../ComparativaF1_Resto.pdf", format="pdf", width=1400, height=600)

In [8]:
fig = px.box(df, y="F1", color="sampler", x="dataset")
fig.show()
fig.write_image("../ComparativaF1_por_dataset.pdf", format="pdf", width=1920)